<a href="https://colab.research.google.com/github/TheNotFire/l/blob/main/ytestn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Install & Run WebUI reForge (Optimized)
import os
import json
import shutil
from google.colab import drive

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# @markdown ### 1. System & Storage
update_repo = True # @param {type:"boolean"}
update_extensions = False # @param {type:"boolean"}
mount_drive = True # @param {type:"boolean"}
output_drive_folder = "android-colab-forge" # @param {type:"string"}

# @markdown ### 2. Authentication
# @markdown Paste your Civitai API Key to download restricted models.
civitai_token = "" # @param {type:"string"}
# @markdown (Optional) HuggingFace Token if downloading gated models.
hf_token = "" # @param {type:"string"}

# Paths
root_dir = "/content"
repo_dir = os.path.join(root_dir, "reForge")
models_dir = os.path.join(repo_dir, "models", "Stable-diffusion")
lora_dir = os.path.join(repo_dir, "models", "Lora")
vae_dir = os.path.join(repo_dir, "models", "VAE")
esrgan_dir = os.path.join(repo_dir, "models", "ESRGAN")
adetailer_dir = os.path.join(repo_dir, "models", "adetailer")
extensions_dir = os.path.join(repo_dir, "extensions")

# Environment Optimizations
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

# ==============================================================================
# 2. INSTALLATION & SETUP
# ==============================================================================
print("Installing dependencies...")
!apt-get -y install -qq aria2 libcairo2-dev

# Clone or Update reForge
if not os.path.exists(repo_dir):
    print("Cloning WebUI reForge...")
    !git clone https://github.com/Panchovix/stable-diffusion-webui-reForge {repo_dir}
else:
    if update_repo:
        print("Updating reForge...")
        !cd {repo_dir} && git pull
    else:
        print("reForge already installed. Skipping update.")

# Mount Drive & Link Outputs
if mount_drive:
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')

    drive_path = os.path.join("/content/drive/MyDrive", output_drive_folder)
    os.makedirs(drive_path, exist_ok=True)

    outputs_dir = os.path.join(repo_dir, "outputs")
    # Clean up local folder if it exists and is not a symlink, then link
    if os.path.exists(outputs_dir) and not os.path.islink(outputs_dir):
        shutil.rmtree(outputs_dir)
    if not os.path.exists(outputs_dir):
        os.symlink(drive_path, outputs_dir)
        print(f"Outputs linked to: {drive_path}")

# ==============================================================================
# 3. EXTENSIONS
# ==============================================================================
print("Checking Extensions...")
extensions = [
    "https://github.com/Anzhc/aadetailer-reforge",
    "https://github.com/otacoo/sd-webui-civitai-downloader",
    "https://github.com/Haoming02/sd-forge-couple",
    "https://github.com/zixaphir/Stable-Diffusion-Webui-Civitai-Helper",
    "https://github.com/zanllp/sd-webui-infinite-image-browsing",
    "https://github.com/DominikDoom/a1111-sd-webui-tagcomplete",
    "https://github.com/catppuccin/stable-diffusion-webui" # Theme
]

for ext_url in extensions:
    ext_name = ext_url.split('/')[-1].replace('.git', '')
    ext_path = os.path.join(extensions_dir, ext_name)
    if not os.path.exists(ext_path):
        !git clone {ext_url} {ext_path}
    elif update_extensions:
        !cd {ext_path} && git pull

# Fix Infinite Image Browsing Permissions
iib_env = os.path.join(extensions_dir, "sd-webui-infinite-image-browsing", ".env")
if os.path.exists(os.path.dirname(iib_env)):
    with open(iib_env, "w") as f:
        f.write("IIB_ACCESS_CONTROL=disable")

# ==============================================================================
# 4. DOWNLOADER LOGIC
# ==============================================================================
def download_file(url, destination_dir, filename=None):
    if not url: return
    os.makedirs(destination_dir, exist_ok=True)

    # Append Civitai Token if needed
    if civitai_token and "civitai.com" in url and "token=" not in url:
        separator = "&" if "?" in url else "?"
        url = f"{url}{separator}token={civitai_token}"

    # HuggingFace Convert Blob -> Resolve
    url = url.replace("blob/main", "resolve/main")

    # Command construction
    base_cmd = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M"
    out_flag = f"-o '{filename}'" if filename else ""
    cmd = f"{base_cmd} {out_flag} '{url}' -d '{destination_dir}'"
    os.system(cmd)

# ==============================================================================
# 5. MODEL DOWNLOADS
# ==============================================================================
# @markdown ### 3. Models
model_map = {
    "None": "",
    "Animagine XL 3.1": "https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors",
    "Rae Diffusion XL v2": "https://huggingface.co/Raelina/Rae-Diffusion-XL-V2/resolve/main/RaeDiffusion-XL-v2.safetensors",
    "Kivotos XL 2.0": "https://huggingface.co/yodayo-ai/kivotos-xl-2.0/resolve/main/kivotos-xl-2.0.safetensors",
    "UrangDiffusion 2.0": "https://huggingface.co/kayfahaarukku/UrangDiffusion-2.0/resolve/main/UrangDiffusion-2.0.safetensors",
    "WAI NSFW Illustrious": "https://civitai.com/api/download/models/1490781?type=Model&format=SafeTensor&size=pruned&fp=fp16",
    "Illustrious XL SmoothFT": "https://civitai.com/api/download/models/1015877?type=Model&format=SafeTensor&size=pruned&fp=fp16",
    "Madly Mix Nightnoob": "https://civitai.com/api/download/models/1202045?type=Model&format=SafeTensor&size=full&fp=fp16",
    "Hassaku XL Illustrious": "https://civitai.com/api/download/models/1240288?type=Model&format=SafeTensor&size=pruned&fp=bf16",
    "Flux.1 Dev (BnB nf4)": "https://huggingface.co/lllyasviel/flux1-dev-bnb-nf4/resolve/main/flux1-dev-bnb-nf4-v2.safetensors"
}
selected_model = "None" # @param ["None", "Animagine XL 3.1", "Rae Diffusion XL v2", "Kivotos XL 2.0", "UrangDiffusion 2.0", "WAI NSFW Illustrious", "Illustrious XL SmoothFT", "Madly Mix Nightnoob", "Hassaku XL Illustrious", "Flux.1 Dev (BnB nf4)"]
custom_model_url = "https://civitai.com/api/download/models/2514310?type=Model&format=SafeTensor&size=pruned&fp=fp16" # @param {type:"string"}

print("Downloading Checkpoints...")
if selected_model != "None":
    download_file(model_map[selected_model], models_dir)
if custom_model_url:
    download_file(custom_model_url, models_dir)

# Fallback download if empty
if not any(f.endswith(('.safetensors', '.ckpt')) for f in os.listdir(models_dir) if os.path.isfile(os.path.join(models_dir, f))):
    print("No models found. Downloading fallback...")
    download_file("https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors", models_dir)

# ==============================================================================
# 6. LoRAs & UTILS
# ==============================================================================
# @markdown ### 4. LoRAs & Extras
download_schnell_flux_lora = False # @param {type:"boolean"}
custom_lora_url = "" # @param {type:"string"}
drive_lora_folder = "" # @param {type:"string"}
load_recursively = True # @param {type:"boolean"}

# Drive LoRAs
if drive_lora_folder.strip():
    p = drive_lora_folder.strip()
    if os.path.exists(p):
        print(f"Linking Drive LoRAs from: {p}")
        folder_name = os.path.basename(p) or "Drive_Loras"
        dest_link = os.path.join(lora_dir, folder_name)
        if not os.path.exists(dest_link):
            if load_recursively:
                os.symlink(p, dest_link)
            else:
                os.makedirs(dest_link, exist_ok=True)
                for f in os.listdir(p):
                    if f.endswith(('.safetensors', '.pt', '.ckpt')):
                        os.symlink(os.path.join(p, f), os.path.join(dest_link, f))

# URL LoRAs
if custom_lora_url:
    for url in custom_lora_url.split(','):
        download_file(url.strip(), lora_dir)
if download_schnell_flux_lora:
    download_file("https://civitai.com/api/download/models/759853?type=Model&format=SafeTensor", lora_dir)

# ADetailer & Upscalers
print("Downloading Support Models...")
ad_list = [
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8n.pt", "face_yolov8n.pt"),
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8n.pt", "hand_yolov8n.pt"),
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/person_yolov8n-seg.pt", "person_yolov8n-seg.pt")
]
for url, name in ad_list:
    download_file(url, adetailer_dir, name)

download_file("https://huggingface.co/Kim2091/AnimeSharp/resolve/main/4x-AnimeSharp.safetensors", esrgan_dir, "4x-AnimeSharp.safetensors")
download_file("https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth", esrgan_dir, "4x-UltraSharp.pth")

# Config Injection
if civitai_token:
    cfg_path = os.path.join(repo_dir, "config.json")
    data = {}
    if os.path.exists(cfg_path):
        try:
            with open(cfg_path, "r") as f: data = json.load(f)
        except: pass
    data["civitai_api_key"] = civitai_token
    with open(cfg_path, "w") as f: json.dump(data, f, indent=4)

# ==============================================================================
# 7. LAUNCH
# ==============================================================================
# @markdown ### 5. Launch
# @markdown Added `--xformers` (Speed) and `--no-half-vae` (Stability for XL) by default.
args = "--share --theme dark --enable-insecure-extension-access --gradio-queue --no-download-sd-model --xformers --no-half-vae" # @param {type:"string"}

%cd {repo_dir}
!python launch.py {args}